### SetUp

In [ ]:
# # load API key from .env 
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# llm model
model="gpt-4o-mini"

### Business Logic

In [8]:
from langchain_classic.chains import SequentialChain
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain
from langchain_openai import ChatOpenAI

# Langchain Model setup
llm = ChatOpenAI(temperature=0.9, model=model)

# Step 1: Poem Generator
poem_prompt = ChatPromptTemplate.from_template(
    "Write a short 4-6 line poem about the theme: {theme}. "
    "Send clear output without additional formatting."
)
chain_poem = LLMChain(llm=llm, prompt=poem_prompt,
                      output_key="poem")

# Step 2: Motivational Quote Generator
quote_prompt = ChatPromptTemplate.from_template(
    "Give a motivational one-liner quote about the theme: {theme}. "
    "Send clear output without additional formatting."
)
chain_quote = LLMChain(llm=llm, prompt=quote_prompt,
                       output_key="quote")

# Step 3: Affirmation Generator
affirm_prompt = ChatPromptTemplate.from_template(
    "Create a positive affirmation (starting with 'I am' or 'I can') "
    "about the theme: {theme}. Send clear output without additional formatting."
)
chain_affirm = LLMChain(llm=llm, prompt=affirm_prompt,
                        output_key="affirmation")

# Step 4: Reflection Prompt Generator
reflect_prompt = ChatPromptTemplate.from_template(
    "Suggest a journaling or self-reflection question based on the theme: {theme}. "
    "Send clear output without additional formatting."
)
chain_reflect = LLMChain(llm=llm, prompt=reflect_prompt,
                         output_key="reflection")

# Overall MindSpark Chain
mindspark_chain = SequentialChain(
    chains=[chain_poem, chain_quote, chain_affirm, chain_reflect],
    input_variables=["theme"],
    output_variables=["poem", "quote", "affirmation", "reflection"],
    verbose=True
)

### Display Functions

In [ ]:
# # Example run
# result = mindspark_chain({"theme": "Goal"})

# def display_inspiration(result):
#     print("🌟 MindSpark Daily Inspiration 🌟")
#     print("\nTheme:", result["theme"])
#     print("\n📜 Poem:\n", result["poem"])
#     print("\n💡 Quote:\n", result["quote"])
#     print("\n✨ Affirmation:\n", result["affirmation"])
#     print("\n🪞 Reflection Prompt:\n", result["reflection"])

# # Example usage
# display_inspiration(result)

### UI

In [9]:
# Add UI for Display
import ipywidgets as widgets
from IPython.display import display

# Chat display area
chat_area = widgets.HTML(value="", layout=widgets.Layout(width="100%", height="300px", overflow="auto", border="1px solid gray", padding="10px"))

# Input + send button
input_box = widgets.Text(placeholder="Enter a theme (e.g., courage, hope)...")
send_button = widgets.Button(description="Generate", button_style="success")

# Function to handle sending
def on_send(_):
    theme = input_box.value.strip()
    if not theme:
        return
    input_box.value = ""

    # Append user message
    chat_area.value += f"<p><b>You:</b> {theme}</p>"

    # Run MindSpark chain
    result = mindspark_chain({"theme": theme})

    # Format bot reply
    bot_reply = f"""
    <p><b>Bot (MindSpark):</b></p>
    <p><b>📜 Poem:</b><br>{result['poem']}</p>
    <p><b>💡 Quote:</b><br>{result['quote']}</p>
    <p><b>✨ Affirmation:</b><br>{result['affirmation']}</p>
    <p><b>🪞 Reflection:</b><br>{result['reflection']}</p>
    """
    chat_area.value += bot_reply

# Bind button click
send_button.on_click(on_send)

# Reset button
reset_button = widgets.Button(description="Reset", button_style="warning")

def on_reset(_):
    chat_area.value = ""
    chat_area.value += "<p><b>Bot (MindSpark):</b> Welcome! Enter a theme to receive your inspiration pack.</p>"

reset_button.on_click(on_reset)

# Layout
ui = widgets.VBox([
    chat_area,
    widgets.HBox([input_box, send_button, reset_button])
])

# Initial greeting
chat_area.value += "<p><b>Bot (MindSpark):</b> Welcome! Enter a theme to receive your inspiration pack.</p>"

display(ui)
